In [0]:
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "aaa_dp")
dbutils.widgets.text("pipeline_id", "")
dbutils.widgets.text("pipeline_name", "DLT_pipeline")
dbutils.widgets.text("report_dir", "/Volumes/aaa_dp/report/report_json")

CATALOG = dbutils.widgets.get("catalog")
PIPELINE_NAME = dbutils.widgets.get("pipeline_name")
REPORT_DIR = dbutils.widgets.get("report_dir")
REPORT_PATH = f"{REPORT_DIR}/pipeline_report.json"


# pipeline and the latest update
w = WorkspaceClient()

def resolve_pipeline_id() -> str:
    pid = dbutils.widgets.get("pipeline_id").strip()
    if pid:
        return pid
    matches = [p for p in w.pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")
               if p.name == PIPELINE_NAME]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one pipeline named '{PIPELINE_NAME}', found {len(matches)}")
    return matches[0].pipeline_id

pipeline_id = resolve_pipeline_id()
updates = w.pipelines.list_updates(pipeline_id=pipeline_id, max_results=1).updates or []
if not updates:
    raise ValueError("No updates found for pipeline")

latest = updates[0]
update_id = latest.update_id
pipeline_state = getattr(latest.state, "value", str(latest.state))
created_ms = getattr(latest, "creation_time", None)

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_id": pipeline_id,
    "update_id": update_id,
    "update_created_utc": datetime.fromtimestamp(created_ms / 1000, tz=timezone.utc).isoformat() if created_ms else None,
    "pipeline_status": pipeline_state,
    "status": "SUCCESS",
    "tables": {},
}

# event log 
events = (spark.sql(f"SELECT * FROM event_log('{pipeline_id}')")
          .select("timestamp", "event_type",
                  F.col("origin.update_id").alias("update_id"),
                  F.col("origin.flow_name").alias("flow_name"),
                  F.col("details").cast("string").alias("details_json"))
          .filter((F.col("update_id") == update_id) & F.col("flow_name").startswith(f"{CATALOG}.")))

events.cache()

def jp(path):
    return F.get_json_object(F.col("details_json"), path)

LABELS = {"MAINTENANCE_TYPE_COMPLETE_RECOMPUTE": "Full recompute",
          "MAINTENANCE_TYPE_NO_OP": "No-op (no changes)"}

try:
    flow_events = events.filter(F.col("event_type") == "flow_progress")

    target_tables = [r["flow_name"] for r in flow_events.select("flow_name").distinct().collect()]

    written = {r["flow_name"]: r["written_rows"] for r in
               flow_events.withColumn("rows", jp("$.flow_progress.metrics.num_output_rows").cast("long"))
                          .filter(F.col("rows").isNotNull())
                          .groupBy("flow_name")
                          .agg(F.max_by("rows", "timestamp").alias("written_rows"))
                          .collect()}

    technique_schema = "array<struct<maintenance_type:string,is_chosen:boolean>>"
    technique = {r["flow_name"]: r["maintenance_type"] for r in
                 events.filter(F.col("event_type") == "planning_information")
                       .withColumn("t", F.from_json(jp("$.planning_information.technique_information"), technique_schema))
                       .select("flow_name", "timestamp", F.explode("t").alias("t"))
                       .filter(F.col("t.is_chosen"))
                       .select("flow_name", "timestamp", F.col("t.maintenance_type").alias("maintenance_type"))
                       .groupBy("flow_name")
                       .agg(F.max_by("maintenance_type", "timestamp").alias("maintenance_type"))
                       .collect()}

    for name in target_tables:
        m_type = technique.get(name)
        entry = {"written_rows": written.get(name),
                 "maintenance_type": m_type,                            # сырое значение из event log
                 "incrementalization": "N/A" if m_type is None else LABELS.get(m_type, "Incremental")}
        try:
            entry.update(status="SUCCESS", total_rows=spark.table(name).count())
        except Exception as e:
            entry.update(status="FAILED", error=str(e)[:200])
        report["tables"][name] = entry

except Exception as e:
    report["status"] = "FAILED"
    report["error"] = str(e)[:300]
finally:
    events.unpersist()

tables_ok = all(t["status"] == "SUCCESS" for t in report["tables"].values())
if report["status"] == "SUCCESS" and not (pipeline_state == "COMPLETED" and tables_ok and report["tables"]):
    report["status"] = "FAILED"

Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_PATH).write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

if report["status"] != "SUCCESS":
    raise RuntimeError("Pipeline report is not clean, see report file")     # чтобы задача job не выглядела успешной